In [19]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType

# 1. Initialize SparkSession
spark = SparkSession.builder \
 .appName("OptimizationAndCachingDemo") \
 .getOrCreate()


In [20]:
# 2. Create a slightly larger DataFrame to make the time difference noticeable (10 million rows)
print("Generating 10 million rows...")
df = spark.range(0, 10000000).withColumn("value", col("id") % 1000)
# ==========================================
# PART A: OPTIMIZATION (Catalyst Optimizer)
# ==========================================
print("\n=========================================")
print("PART A: CATALYST OPTIMIZER PLAN")
print("=========================================")
# We apply multiple transformations.
# Spark's Catalyst Optimizer will combine these efficiently under the hood.
optimized_df = df.filter(col("value") > 500).filter(col("id") <
5000000).select("id", "value")
# .explain() prints the physical execution plan directly to the console/terminal
print("Physical Execution Plan (Notice how it pushes filters down):")
optimized_df.explain()

Generating 10 million rows...

PART A: CATALYST OPTIMIZER PLAN
Physical Execution Plan (Notice how it pushes filters down):
== Physical Plan ==
*(1) Project [id#494L, (id#494L % 1000) AS value#496L]
+- *(1) Filter (((id#494L % 1000) > 500) AND (id#494L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [21]:
# ==========================================
# PART B: CACHING & PERFORMANCE TEST
# ==========================================
print("\n=========================================")
print("PART B: CACHING EXECUTION TIMES")
print("=========================================")
# Test 1: Execution WITHOUT caching
start_time = time.time()
count_uncached = optimized_df.count() # Action triggers DAG
end_time = time.time()
print(f"1. UNCACHED Execution | Count: {count_uncached} | Time:{round(end_time - start_time, 4)} seconds")
# Cache the DataFrame
print("\n--> Applying .cache() (This is lazy, nothing happens yet)...")
optimized_df.cache()


PART B: CACHING EXECUTION TIMES
1. UNCACHED Execution | Count: 2495000 | Time:0.2407 seconds

--> Applying .cache() (This is lazy, nothing happens yet)...


DataFrame[id: bigint, value: bigint]

In [22]:
# Test 2: First Execution WITH caching
# (This takes time because it has to calculate the data AND write it to RAM)
start_time = time.time()
count_first_cache = optimized_df.count() # Action triggers caching
end_time = time.time()
print(f"2. FIRST CACHED Action | Count: {count_first_cache} | Time:{round(end_time - start_time, 4)} seconds (Materializing cache)")

[Stage 3:=============================>                             (1 + 1) / 2]

2. FIRST CACHED Action | Count: 2495000 | Time:1.7187 seconds (Materializing cache)


In [23]:
# Test 3: Second Execution WITH caching
# (This will be lightning fast because the data is already sitting in RAM)
start_time = time.time()
count_second_cache = optimized_df.count()
end_time = time.time()
print(f"3. SECOND CACHED Action| Count: {count_second_cache} | Time:{round(end_time - start_time, 4)} seconds (Read from memory)")

3. SECOND CACHED Action| Count: 2495000 | Time:0.187 seconds (Read from memory)


In [24]:
# Clean up: Remove from memory when done
optimized_df.unpersist()
print("\n--> Unpersisted DataFrame to free up memory.")


--> Unpersisted DataFrame to free up memory.
